# Fase 3 | Nucleo algoritmico, eficiencia y POO
Continuacion de F1/F2 con telemetria real MetroPT-3.

## Entrada: dataset limpio y trazabilidad

In [3]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.pipeline_real import *
RUTA=ROOT/'datos'/'MetroPT3(AirCompressor).csv'
print('Raiz:',ROOT)

df=cargar(RUTA)
limpios,rechazados,audit=procesar(df)
print(audit)

Raiz: C:\Users\pamel\OneDrive\Escritorio\UNAB_Grupo10_GitHub\UNAB_Grupo10_Corregido
{'originales': 1516948, 'aceptados': 1516948, 'rechazados': 0}


## Pregunta algoritmica
Comparar el recuento de observaciones con presion TP2 sobre un umbral empirico (percentil 95). Este indicador no representa una falla confirmada.

In [4]:
import numpy as np
serie=limpios['TP2'].to_numpy(dtype=float)
umbral=float(np.quantile(serie,.95))
print('Umbral percentil 95:',umbral)

Umbral percentil 95: 9.59


## Programacion estructurada, recursiva y vectorizada
Iterativo O(n) y espacio auxiliar O(1); recursivo divide y venceras O(n), memoria adicional por particiones/vistas y pila O(log n); vectorizado O(n) con mascara temporal O(n).

In [5]:
muestra=serie[:min(20000,len(serie))]
resultados=[contar_iterativo(muestra,umbral),contar_recursivo(muestra,umbral),contar_vectorizado(muestra,umbral)]
print(resultados)
if len(set(resultados))!=1: raise ValueError('Resultados inconsistentes')

[514, 514, 514]


## Benchmark reproducible con timeit
Los tiempos dependen del equipo y del entorno. Se repiten ejecuciones con igual muestra y umbral.

In [6]:
import timeit, pandas as pd
medidas=[]
for n in [100,1000,5000,10000]:
 a=serie[:min(n,len(serie))]
 for nombre,fn in [('iterativo',contar_iterativo),('recursivo',contar_recursivo),('vectorizado',contar_vectorizado)]:
  t=timeit.repeat(lambda:fn(a,umbral),repeat=5,number=3)
  medidas.append({'n':len(a),'algoritmo':nombre,'segundos_por_ejecucion':min(t)/3})
bench=pd.DataFrame(medidas)
display(bench)

,n,algoritmo,segundos_por_ejecucion
0,100,iterativo,0.000008
1,100,recursivo,0.000122
2,100,vectorizado,0.000003
3,1000,iterativo,0.000069
4,1000,recursivo,0.001203
5,1000,vectorizado,0.000003
6,5000,iterativo,0.000335
7,5000,recursivo,0.006090
8,5000,vectorizado,0.000005
9,10000,iterativo,0.000681


## Interpretacion y optimizacion
Se elige la implementacion segun tiempos medidos y uso de memoria, no por suposicion. La recursividad se incluye como evidencia academica, no como recomendacion automatica.

## POO: encapsulamiento, herencia y polimorfismo
`EstrategiaResumen` define interfaz; `ResumenMedias` y `ResumenMedianas` implementan comportamientos intercambiables; `Analizador` encapsula la estrategia.

In [7]:
a=Analizador(ResumenMedias()); print('Media TP2:',a.ejecutar(limpios)['TP2'])
a.cambiar_estrategia(ResumenMedianas()); print('Mediana TP2:',a.ejecutar(limpios)['TP2'])

Media TP2: 1.3678259663482208
Mediana TP2: -0.0120000000000004


## Resultados descriptivos por estado de control

In [8]:
display(limpios.groupby('COMP')[ANALOGAS].agg(['count','mean','median']).round(3))

TP2                    TP3                     H1                \
        count   mean median    count   mean median    count   mean median   
COMP                                                                        
0.0    247328  8.406  8.944   247328  8.835  8.796   247328  0.232 -0.012   
1.0   1269620 -0.003 -0.012  1269620  9.014  8.982  1269620  8.997  8.968   

     DV_pressure  ...        Reservoirs               Oil_temperature          \
           count  ... median      count   mean median           count    mean   
COMP              ...                                                           
0.0       247328  ... -0.018     247328  8.834  8.794          247328  66.094   
1.0      1269620  ... -0.020    1269620  9.015  8.984         1269620  61.972   

             Motor_current                
      median         count   mean median  
COMP                                      
0.0   65.575        247328  5.604  5.828  
1.0   62.075       1269620  1.358  0.042  

[2 rows x 21 columns]

## Verificacion y cierre
Ejecutar pruebas con `python -m pytest -q`, guardar resultados reales y actualizar informe. No hay etiquetas de falla por fila; las diferencias observadas son descriptivas, no causales.

In [9]:
print('Registros:',len(limpios),'Pruebas de equivalencia:',len(set(resultados))==1)

Registros: 1516948 Pruebas de equivalencia: True


In [10]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=str(ROOT)
)


CompletedProcess(args=['C:\\Users\\pamel\\miniconda3\\envs\\magister_unab\\python.exe', '-m', 'pytest', '-q', 'tests'], returncode=0)

In [11]:
resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests"],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)

============================= test session starts =============================
platform win32 -- Python 3.11.18, pytest-8.4.2, pluggy-1.7.0 -- C:\Users\pamel\miniconda3\envs\magister_unab\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\pamel\OneDrive\Escritorio\UNAB_Grupo10_GitHub\UNAB_Grupo10_Corregido
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 4 items

tests/test_pipeline_real.py::test_algoritmos PASSED                      [ 25%]
tests/test_pipeline_real.py::test_vacio PASSED                           [ 50%]
tests/test_pipeline_real.py::test_conciliacion PASSED                    [ 75%]
tests/test_pipeline_real.py::test_sin_archivo PASSED                     [100%]

============================== 4 passed in 0.31s ==============================


